In [1]:
import os
os.chdir(os.path.dirname(os.getcwd()))

In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style and size
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [3]:
# Define data path
data_path = 'data/m5-forecasting-accuracy/'

# Load datasets
calendar = pd.read_csv(f"{data_path}calendar.csv")
selling_prices = pd.read_csv(f"{data_path}sell_prices.csv")
sample_submission = pd.read_csv(f"{data_path}sample_submission.csv")
sales = pd.read_csv(f"{data_path}sales_train_evaluation.csv") 

In [4]:
calendar['date'] = pd.to_datetime(calendar['date'])
calendar['dayofmonth'] = calendar['date'].dt.day
calendar = calendar.drop(columns=['weekday'], errors='ignore')
calendar['d'] = calendar['d'].str[2:].astype(int)

for col in ['event_name_1', 'event_type_1']:
    calendar[col] = calendar[col].astype('category').cat.codes

In [5]:
selling_prices.sort_values(by=['store_id', 'item_id', 'wm_yr_wk'], inplace=True)
selling_prices['sell_price_rel_diff'] = selling_prices.groupby(['store_id', 'item_id'])['sell_price'].pct_change()

def cumrel(x):
    return (x - x.cummin()) / (1 + x.cummax() - x.cummin())

selling_prices['sell_price_cumrel'] = selling_prices.groupby(['store_id', 'item_id'])['sell_price'].transform(cumrel)
selling_prices['sell_price_roll_sd7'] = selling_prices.groupby(['store_id', 'item_id'])['sell_price'].transform(lambda x: x.rolling(window=7).std())

In [6]:
sales['id'] = sales['id'].str.replace('_validation', '', regex=False)

In [ ]:
n_item = 3049
n_stores = 10
n_all = n_item * n_stores
H = 1  # Assuming 1-step ahead, adjust if needed

# Placeholder for forecasts: shape = (n_all, H, 3 quantiles or params)
PRED = np.empty((n_all, H, 3))

# Sample 80 random item indices
np.random.seed(1234)
do_items = np.sort(np.random.choice(np.arange(1, n_item + 1), 80, replace=False))

In [ ]:
# Step 2: Create future horizon columns like 'd_1942', 'd_1943', ...
H = 1
dm = 1941
future_cols = [f'd_{dm + i + 1}' for i in range(H)]
empty_dt = pd.DataFrame(np.nan, index=sales.index, columns=future_cols)

# Step 3: Append new empty forecast horizon columns to sales
sales = pd.concat([sales, empty_dt], axis=1)

# Step 4: Reshape to long format
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
sales_long = sales.melt(id_vars=id_vars, var_name='d', value_name='demand')

# Step 5: Convert "d_1942" → 1942
sales_long['d'] = sales_long['d'].str.extract(r'd_(\d+)').astype(int)

In [ ]:
n_dates = 1941
iitem = list(range(n_item))  # 0 to n_item-1
iin = len(iitem)

# Filter to the first n_item unique items
filtered_ids = sales_long[sales_long['item_id'].isin(sales['item_id'].unique()[iitem])]
sales_filtered = filtered_ids.copy()

In [10]:
sales

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3_evaluation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,0,0,2,2,...,1,0,3,0,1,1,0,0,1,1
30486,FOODS_3_824_WI_3_evaluation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
30487,FOODS_3_825_WI_3_evaluation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,0,6,0,2,...,0,0,1,2,0,1,0,1,0,2
30488,FOODS_3_826_WI_3_evaluation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,1,1,1,4,6,0,1,1,1,0


In [11]:
sales['demand_store_id'] = sales.groupby(['store_id', 'd'])['demand'].transform('mean')
sales['demand_item_id'] = sales.groupby(['item_id', 'd'])['demand'].transform('mean')
sales['demand_all_id'] = sales.groupby('d')['demand'].transform('mean')

KeyError: 'd'